In [ ]:
import os, json
from itertools import product

RAW_PATH = "../data/benchmark/MT745584.txt"   
TX_PATH  = "../data/processed/covid_kmer.txt"           # Sinh transactions
MAP_PATH = "../data/processed/covid_kmer_map.json"      # bảng id <-> k-mer
K      = 6      # độ dài k-mer (item)
WINDOW = 100    # độ dài mỗi đoạn genome = 1 giao dịch
STRIDE = WINDOW - K + 1     # bước trượt (STRIDE < WINDOW: các đoạn chồng lấn)
MINSUP = 0.1

# 1. Đọc genome
with open(RAW_PATH) as f:
    genome = "".join(l.strip().upper() for l in f if l.strip() and not l.startswith(">"))
print(f"Độ dài genome: {len(genome):,} nucleotide")

# 2. Bảng ánh xạ k-mer -> số nguyên 
kmer2id = {"".join(p): i for i, p in enumerate(product("ACGT", repeat=K), 1)}

# 3. Mỗi đoạn trượt -> một giao dịch gồm tập id các k-mer xuất hiện trong đoạn
starts, n_skip = [], 0
with open(TX_PATH, "w") as out:
    for s in range(0, len(genome) - WINDOW + 1, STRIDE):
        w = genome[s:s + WINDOW]
        ids = {kmer2id[w[j:j + K]] for j in range(WINDOW - K + 1) if w[j:j + K] in kmer2id}
        if not ids:
            n_skip += 1
            continue
        out.write(" ".join(map(str, sorted(ids))) + "\n")
        starts.append(s)

with open(MAP_PATH, "w") as f:
    json.dump({"k": K, "window": WINDOW, "stride": STRIDE, "starts": starts,
               "id2kmer": {str(v): k for k, v in kmer2id.items()}}, f)

print(f"Số giao dịch: {len(starts)} (bỏ {n_skip} đoạn rỗng) | k={K}, window={WINDOW}, stride={STRIDE}")
print(f"Đã ghi {TX_PATH}")

Độ dài genome: 29,860 nucleotide
Số giao dịch: 314 (bỏ 0 đoạn rỗng) | k=6, window=100, stride=95
Đã ghi ../data/processed/covid_kmer.txt


In [9]:
!julia --project=.. ../run_cli.jl --input {TX_PATH} --minsup {MINSUP} --algo optimized --output ../results/covid_out.txt

[1/3] Đang đọc CSDL từ: ../data/processed/covid_kmer.txt
[2/3] Đang chạy thuật toán Apriori (optimized) | Minsup tương đối: 0.1 (Tối thiểu 32 giao dịch)
[3/3] Hoàn thành trong: 1.429s. Tìm thấy 12 frequent itemsets.
Đang ghi kết quả ra: ../results/covid_out.txt


  Activating project at `d:\HCMUS\Data Mining\Frequent Itemset\Frequent Itemset\21`


In [10]:
import os, json

output_path = "../results/covid_out.txt"
decoded_path = "../results/covid_out_decoded.txt"
info = json.load(open("../data/processed/covid_kmer_map.json"))
id2kmer, starts, W = info["id2kmer"], info["starts"], info["window"]
N = len(starts)
MIN_SIZE = 1     # đổi thành 2 khi đã hạ minsup và muốn chỉ xét tổ hợp >= 2 k-mer
TOP = 20         # số dòng in ra

if not os.path.exists(output_path):
    print(f"Chưa tìm thấy {output_path}. Hãy chạy các cell trước!")
else:
    # 1. Đọc & giải mã: mỗi itemset -> (list id, chuỗi k-mer, support)
    frequent_itemsets = []
    with open(output_path) as f:
        for line in f:
            if "#SUP:" in line:
                left, sup = line.strip().split("#SUP:")
                ids = left.split()
                frequent_itemsets.append((ids, " + ".join(id2kmer[i] for i in ids), int(sup)))

    by_size = {}
    for ids, _, _ in frequent_itemsets:
        by_size[len(ids)] = by_size.get(len(ids), 0) + 1
    print(f"Tổng số itemset phổ biến: {len(frequent_itemsets)} | theo kích thước: {dict(sorted(by_size.items()))}")

    # 2. Lọc theo kích thước, xếp theo support giảm dần
    cands = sorted((x for x in frequent_itemsets if len(x[0]) >= MIN_SIZE),
                   key=lambda x: (-x[2], -len(x[0])))

    # 3. In bảng: #(id) - mã - support
    print("-" * 70)
    print(f"{'STT':<4} | {'#(id)':<18} | {'Mã (k-mer)':<24} | {'Support':<7} | {'% đoạn':<6}")
    print("-" * 70)
    for idx, (ids, motif, sup) in enumerate(cands[:TOP], 1):
        id_str = "#" + "+".join(ids)
        print(f"{idx:<4} | {id_str:<18} | {motif:<24} | {sup:<7} | {100 * sup / N:5.1f}%")

    # 4. Ghi file đã giải mã: "#id - mã - support"
    with open(decoded_path, "w") as f:
        for ids, motif, sup in cands:
            f.write(f"#{'+'.join(ids)} - {motif} - {sup}\n")
    print(f"\nĐã ghi {len(cands)} dòng vào {decoded_path}")

    # 5. Vị trí trên genome của motif đứng đầu (để thảo luận ý nghĩa sinh học)
    if cands:
        top = set(cands[0][0])
        pos = []
        with open("../data/processed/covid_kmer.txt") as f:
            for s, line in zip(starts, f):
                if top <= set(line.split()):
                    pos.append(s)
        print(f"Motif top-1 ({cands[0][1]}) xuất hiện ở {len(pos)} đoạn, "
              f"vị trí genome từ {min(pos)} đến {max(pos) + W}")

Tổng số itemset phổ biến: 12 | theo kích thước: {1: 12}
----------------------------------------------------------------------
STT  | #(id)              | Mã (k-mer)               | Support | % đoạn
----------------------------------------------------------------------
1    | #3825              | TGTTAA                   | 41      |  13.1%
2    | #4029              | TTGTTA                   | 38      |  12.1%
3    | #4081              | TTTTAA                   | 34      |  10.8%
4    | #129               | AAGAAA                   | 33      |  10.5%
5    | #2800              | GGTGTT                   | 33      |  10.5%
6    | #3836              | TGTTGT                   | 33      |  10.5%
7    | #3857              | TTACAA                   | 33      |  10.5%
8    | #1008              | ATTGTT                   | 32      |  10.2%
9    | #2047              | CTTTTG                   | 32      |  10.2%
10   | #3840              | TGTTTT                   | 32      |  10.2%
11   | #38